# SFT Dataset Preparation for Information Hunter
This notebook processes the input synthetic dataset and applies probabilistic
data-mixing, swapping, and formatting to prepare the SFT dataset.

In [ ]:
import os
import sys
import random
import pandas as pd
from tqdm import tqdm


# Ensure stdout uses UTF-8 to prevent encoding errors on Windows terminals
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

In [ ]:
class Config:
    # ------------------------------------------------------------
    # CONFIGURATION PARAMETERS (PROBABILITIES WILL BE NORMALIZED)
    # ------------------------------------------------------------
    # Relative weights/probabilities of the 4 main context generation operations
    P_BASELINE = 0.3          # Clean original context
    P_SWAPPED = 0.15           # Completely unrelated single context -> output fallback
    P_MIXED_WITH_TARGET = 0.40  # True context mixed with random distractors
    P_MIXED_NO_TARGET = 0.15    # Random distractors only (no true context) -> output fallback

    # Probability to apply RAG formatting (e.g. [তথ্যসূত্র N]: ...) to any chunk
    P_RAG_FORMAT = 0.25

    # When RAG formatting is applied, probability of selecting QA-pair style vs Doc-style
    # If QA-style is selected but no QA pairs are available, it falls back to Doc-style.
    P_QA_FORMAT = 0.50

    # Probability that the exact target QA pair (question/answer being queried)
    # is included when the target chunk is formatted in QA-pair style.
    # If excluded, the answer is NOT present and output will be the fallback.
    P_INCLUDE_TARGET_QA = 0.6

    # Range of chunks to mix in mixed context modes (min 2, max 5)
    MIN_MIXED_CHUNKS = 2
    MAX_MIXED_CHUNKS = 5

    # Output fallback text when answer is not in the context
    NO_INFO_FALLBACK = "[তথ্য পাওয়া যায়নি]"

    # Random seed for reproducibility
    SEED = 42

    # Paths
    INPUT_CSV_PATH = r"d:\ALL CODES\IUT Datathon\Training\Make dataset\synthetic_bangla_dataset.csv"
    OUTPUT_CSV_PATH = r"d:\ALL CODES\IUT Datathon\Barno's Lab\DUAL MODEL\repopulate datase\augmented_sft_dataset.csv"

In [ ]:
# Global System Instruction for SFT
INSTRUCTION = (
    "You will be provided with a question and some relevant reference contexts. "
    "Using the provided contexts, extract the correct answer to the question verbatim. "
    "If the answer is not present in the contexts, output exactly '[তথ্য পাওয়া যায়নি]'. "
    "Do not include any extra explanations, details, or formatting."
)

def build_sft_prompt(context_text, question):
    """Formats the input prompt for SFT."""
    return (
        f"Instruction: {INSTRUCTION}\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )

In [ ]:
def normalize_probabilities():
    """Normalizes the main operation weights to sum to 1.0."""
    total = Config.P_BASELINE + Config.P_SWAPPED + Config.P_MIXED_WITH_TARGET + Config.P_MIXED_NO_TARGET
    p_base = Config.P_BASELINE / total
    p_swap = Config.P_SWAPPED / total
    p_mix_t = Config.P_MIXED_WITH_TARGET / total
    p_mix_nt = Config.P_MIXED_NO_TARGET / total
    return [p_base, p_swap, p_mix_t, p_mix_nt]

def format_chunk(title, context, qa_pairs, apply_rag=True, p_qa=0.5, target_qa=None, include_target=True):
    """
    Formats a context chunk.
    If apply_rag is True:
        Probabilistically formats as QA-style list or Doc-style.
    Else:
        Returns raw context block.
    Returns:
        (formatted_text, is_target_present)
    """
    if not apply_rag:
        return context, True

    # Try QA-pair style format with probability p_qa
    if random.random() < p_qa and len(qa_pairs) > 0:
        sibling_pairs = [p for p in qa_pairs if p != target_qa]
        formatted_qa = []
        is_target_present = False
        
        if target_qa is not None:
            if include_target:
                # Include target QA and sample up to 2 siblings
                sample_size = min(2, len(sibling_pairs))
                sampled = random.sample(sibling_pairs, sample_size)
                sampled.append(target_qa)
                random.shuffle(sampled)
                for q, a in sampled:
                    formatted_qa.append(f"Question: {q}\nAnswer: {a}")
                is_target_present = True
            else:
                # Exclude target QA, sample up to 3 siblings
                if len(sibling_pairs) == 0:
                    # Fallback to Doc-style since we have no sibling QA pairs to show
                    return f"Title: {title}\nContent: {context}", True
                sample_size = min(3, len(sibling_pairs))
                sampled = random.sample(sibling_pairs, sample_size)
                random.shuffle(sampled)
                for q, a in sampled:
                    formatted_qa.append(f"Question: {q}\nAnswer: {a}")
                is_target_present = False
        else:
            # Distractor chunk: sample up to 3 pairs from available QA list
            sample_size = min(3, len(qa_pairs))
            sampled = random.sample(qa_pairs, sample_size)
            random.shuffle(sampled)
            for q, a in sampled:
                formatted_qa.append(f"Question: {q}\nAnswer: {a}")
            is_target_present = False
            
        return "\n".join(formatted_qa), is_target_present
    else:
        # Fallback/default Doc-style
        is_target_present = (target_qa is not None)
        return f"Title: {title}\nContent: {context}", is_target_present

In [ ]:
def main():
    # Set random seed
    random.seed(Config.SEED)

    # 1. Load dataset
    print(f"Loading source dataset from: {Config.INPUT_CSV_PATH}")
    if not os.path.exists(Config.INPUT_CSV_PATH):
        raise FileNotFoundError(f"Source file not found at {Config.INPUT_CSV_PATH}")
    df = pd.read_csv(Config.INPUT_CSV_PATH)
    print(f"Loaded {len(df)} source QA pairs.")

    # 2. Build lookup maps
    print("Building lookup maps for titles and QA pairs...")
    title_to_qa = {}
    title_to_contexts = {}
    all_titles = df['title'].dropna().unique().tolist()

    for idx, row in df.iterrows():
        t = row['title']
        q = row['question']
        a = row['correct_answer']
        c = row['context']
        
        if pd.isna(t) or pd.isna(q) or pd.isna(a) or pd.isna(c):
            continue
            
        t = str(t).strip()
        q = str(q).strip()
        a = str(a).strip()
        c = str(c).strip()

        # Build QA pair lookup
        if t not in title_to_qa:
            title_to_qa[t] = []
        # Keep track of QA pairs generated from this article
        if (q, a) not in title_to_qa[t]:
            title_to_qa[t].append((q, a))

        # Build Context lookup
        if t not in title_to_contexts:
            title_to_contexts[t] = []
        if c not in title_to_contexts[t]:
            title_to_contexts[t].append(c)

    # Clean up empty entries
    all_titles = [t for t in all_titles if t in title_to_contexts and title_to_contexts[t]]

    # Normalize operation probabilities
    ops = ['baseline', 'swapped', 'mixed_with_target', 'mixed_no_target']
    probs = normalize_probabilities()
    print(f"Normalized probabilities: Baseline={probs[0]:.2f}, Swapped={probs[1]:.2f}, Mixed-Target={probs[2]:.2f}, Mixed-No-Target={probs[3]:.2f}")

    augmented_data = []
    stats = {op: 0 for op in ops}
    format_stats = {"raw": 0, "rag_doc": 0, "rag_qa": 0}

    # Helper function to sample a distractor context from a different title
    def sample_distractor(exclude_title):
        candidates = [t for t in all_titles if t != exclude_title]
        if not candidates:
            candidates = all_titles # Fallback if only 1 title exists
        dist_title = random.choice(candidates)
        dist_context = random.choice(title_to_contexts[dist_title])
        dist_qa = title_to_qa.get(dist_title, [])
        return dist_title, dist_context, dist_qa

    print("Generating augmented SFT samples...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        target_title = row['title']
        target_question = row['question']
        target_answer = row['correct_answer']
        target_context = row['context']

        if pd.isna(target_title) or pd.isna(target_question) or pd.isna(target_answer) or pd.isna(target_context):
            continue

        target_title = str(target_title).strip()
        target_question = str(target_question).strip()
        target_answer = str(target_answer).strip()
        target_context = str(target_context).strip()

        # Decide operation
        op = random.choices(ops, weights=probs, k=1)[0]
        stats[op] += 1

        context_chunks = []
        expected_output = target_answer

        # Flag for RAG formatting
        apply_rag = random.random() < Config.P_RAG_FORMAT

        if op == 'baseline':
            # Baseline: Just format the target context
            qa_list = title_to_qa.get(target_title, [])
            target_qa = (target_question, target_answer)
            include_target = random.random() < Config.P_INCLUDE_TARGET_QA
            formatted, is_present = format_chunk(
                target_title, target_context, qa_list, 
                apply_rag=apply_rag, p_qa=Config.P_QA_FORMAT,
                target_qa=target_qa, include_target=include_target
            )
            
            # Log formatting statistics
            if not apply_rag:
                format_stats["raw"] += 1
            elif "Answer:" in formatted:
                format_stats["rag_qa"] += 1
            else:
                format_stats["rag_doc"] += 1
                
            context_chunks.append(formatted)
            expected_output = target_answer if is_present else Config.NO_INFO_FALLBACK

        elif op == 'swapped':
            # Swapped: Single distractor context, output is fallback
            dist_title, dist_context, dist_qa = sample_distractor(target_title)
            formatted, _ = format_chunk(
                dist_title, dist_context, dist_qa, 
                apply_rag=apply_rag, p_qa=Config.P_QA_FORMAT,
                target_qa=None
            )
            
            if not apply_rag:
                format_stats["raw"] += 1
            elif "Answer:" in formatted:
                format_stats["rag_qa"] += 1
            else:
                format_stats["rag_doc"] += 1

            context_chunks.append(formatted)
            expected_output = Config.NO_INFO_FALLBACK

        elif op == 'mixed_with_target':
            # Mixed with Target: Concatenate target with K-1 distractors
            num_chunks = random.randint(Config.MIN_MIXED_CHUNKS, Config.MAX_MIXED_CHUNKS)
            
            # 1. Add target chunk
            target_qa_list = title_to_qa.get(target_title, [])
            target_qa = (target_question, target_answer)
            include_target = random.random() < Config.P_INCLUDE_TARGET_QA
            target_formatted, is_present = format_chunk(
                target_title, target_context, target_qa_list, 
                apply_rag=apply_rag, p_qa=Config.P_QA_FORMAT,
                target_qa=target_qa, include_target=include_target
            )
            context_chunks.append(target_formatted)
            
            if not apply_rag:
                format_stats["raw"] += 1
            elif "Answer:" in target_formatted:
                format_stats["rag_qa"] += 1
            else:
                format_stats["rag_doc"] += 1

            # 2. Add K-1 distractors (ensuring unique distractor titles)
            used_titles = {target_title}
            for _ in range(num_chunks - 1):
                candidates = [t for t in all_titles if t not in used_titles]
                if not candidates:
                    candidates = all_titles
                dist_title = random.choice(candidates)
                used_titles.add(dist_title)
                
                dist_context = random.choice(title_to_contexts[dist_title])
                dist_qa = title_to_qa.get(dist_title, [])
                dist_formatted, _ = format_chunk(
                    dist_title, dist_context, dist_qa, 
                    apply_rag=apply_rag, p_qa=Config.P_QA_FORMAT,
                    target_qa=None
                )
                context_chunks.append(dist_formatted)
                
                if not apply_rag:
                    format_stats["raw"] += 1
                elif "Answer:" in dist_formatted:
                    format_stats["rag_qa"] += 1
                else:
                    format_stats["rag_doc"] += 1

            # Shuffle the order of retrieved chunks to prevent positional bias
            random.shuffle(context_chunks)
            expected_output = target_answer if is_present else Config.NO_INFO_FALLBACK

        elif op == 'mixed_no_target':
            # Mixed without Target: Concatenate K distractors, no target
            num_chunks = random.randint(Config.MIN_MIXED_CHUNKS, Config.MAX_MIXED_CHUNKS)
            
            used_titles = {target_title} # Exclude target title
            for _ in range(num_chunks):
                candidates = [t for t in all_titles if t not in used_titles]
                if not candidates:
                    candidates = all_titles
                dist_title = random.choice(candidates)
                used_titles.add(dist_title)
                
                dist_context = random.choice(title_to_contexts[dist_title])
                dist_qa = title_to_qa.get(dist_title, [])
                dist_formatted, _ = format_chunk(
                    dist_title, dist_context, dist_qa, 
                    apply_rag=apply_rag, p_qa=Config.P_QA_FORMAT,
                    target_qa=None
                )
                context_chunks.append(dist_formatted)
                
                if not apply_rag:
                    format_stats["raw"] += 1
                elif "Answer:" in dist_formatted:
                    format_stats["rag_qa"] += 1
                else:
                    format_stats["rag_doc"] += 1

            random.shuffle(context_chunks)
            expected_output = Config.NO_INFO_FALLBACK

        # 3. Apply RAG [Reference N]: prefixes to each chunk if RAG formatting is enabled
        final_context_parts = []
        for i, chunk in enumerate(context_chunks):
            if apply_rag:
                final_context_parts.append(f"[Reference {i+1}]: {chunk}")
            else:
                final_context_parts.append(chunk)

        final_context_text = "\n\n".join(final_context_parts)

        # 4. Construct SFT prompt input and output response
        sft_input = build_sft_prompt(final_context_text, target_question)
        
        augmented_data.append({
            "instruction": INSTRUCTION,
            "input": sft_input,
            "output": expected_output
        })

    # Save augmented SFT dataset
    out_df = pd.DataFrame(augmented_data)
    os.makedirs(os.path.dirname(Config.OUTPUT_CSV_PATH), exist_ok=True)
    out_df.to_csv(Config.OUTPUT_CSV_PATH, index=False)
    print(f"\nSuccessfully generated and saved SFT dataset to: {Config.OUTPUT_CSV_PATH}")
    print(f"Total SFT samples: {len(out_df)}")

    # Print generation statistics
    print("\n" + "="*40)
    print("         DATASET STATISTICS")
    print("="*40)
    for op, count in stats.items():
        pct = (count / len(out_df)) * 100
        print(f"Operation '{op}': {count} ({pct:.2f}%)")
    print("-"*40)
    total_formatted = sum(format_stats.values())
    for ftype, count in format_stats.items():
        pct = (count / total_formatted) * 100 if total_formatted else 0
        print(f"Format style '{ftype}': {count} ({pct:.2f}%)")
    print("="*40)

    # Print a few samples for validation
    print("\n--- SAMPLE GENERATIONS ---")
    samples = out_df.sample(min(2, len(out_df)))
    for idx, sample in samples.iterrows():
        print(f"\n[Sample #{idx}]")
        print(f"--- INPUT ---\n{sample['input']}")
        print(f"--- EXPECTED OUTPUT ---\n{sample['output']}")
        print("="*60)

In [ ]:
if __name__ == "__main__":
    main()